# NBA Lineup RAG Evaluation Report - Part 2

**Course**: Generative AI 424

**Assignment**: Assignment 1

**Name**: Yu-Sen Wu



### ⚠️ Important: Please Set Up the Environment Before Starting

### Step 1: Activate Virtual Environment and Install Dependencies

Run the following commands in your terminal:

```bash
# Enter the project root directory
cd /path/to/nba_lineup_rag

# Create virtual environment (if not already created)
python -m venv venv

# Activate virtual environment
source venv/bin/activate  # macOS/Linux
# or venv\Scripts\activate  # Windows

# Install required packages
pip install -r requirements.txt

# Install Jupyter support (to allow the virtual environment to be used with Jupyter)
pip install ipykernel

# Register the virtual environment as a Jupyter kernel
python -m ipykernel install --user --name=nba_rag --display-name="Python (nba_rag)"
```

### Step 2: Select the Correct Kernel

In Jupyter:
1. Click the kernel name in the upper right corner
2. Choose **"Python (nba_rag)"**
3. Make sure the kernel has switched successfully

### Step 3: Verify the Environment

Run the following "Environment Setup" cell to verify that the environment is properly configured.


## Table of Contents
1. [Environment Setup](#1-環境設置)
2. [Method Description](#2-方法說明)
3. [Test Questions](#3-測試問題)
4. [Evaluation Results](#4-評估結果)
5. [Comparative Analysis](#5-比較分析)
6. [Conclusion](#6-結論)

## 1. Environment Setup

In [1]:
# Set up path
import sys
from pathlib import Path

# Add project root directory to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root directory: {project_root}")

Project root directory: /Users/wuyusen/Desktop/Northwestern MLDS/424 Gen AI/hw/nba_lineup_rag


In [2]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Check API key
import os
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f"✅ OPENAI_API_KEY is set (first 10 chars: {api_key[:10]}...)")
else:
    print("❌ OPENAI_API_KEY is not set. Please set it in the .env file.")

✅ OPENAI_API_KEY is set (first 10 chars: sk-proj-jh...)


In [3]:
# Import RAG modules
from src.rag.langchain_rag import LangChainRAG
from src.rag.hyde import HyDERetriever
from src.rag.reranker import RerankedRAG

print("✅ RAG modules loaded successfully")

✅ RAG modules loaded successfully


In [4]:
# Initialize each RAG system
print("Initializing LangChain RAG...")
langchain_rag = LangChainRAG(model="gpt-4o-mini")

print("Initializing HyDE Retriever...")
hyde_retriever = HyDERetriever()

print("Initializing Reranked RAG...")
reranked_rag = RerankedRAG()

print("\n✅ All RAG systems successfully initialized")

Initializing LangChain RAG...
2026-02-03 14:41:59 | src.vectordb.chroma_client | INFO     | chroma_client.py:77 | Connecting to ChromaDB: /Users/wuyusen/Desktop/Northwestern MLDS/424 Gen AI/hw/nba_lineup_rag/data/chroma
2026-02-03 14:41:59 | src.vectordb.chroma_client | INFO     | chroma_client.py:89 | Successfully connected to ChromaDB
2026-02-03 14:41:59 | src.embeddings.e5_embedder | INFO     | e5_embedder.py:112 | Loading E5 model: intfloat/e5-large-v2 (device: cpu)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-02-03 14:42:00 | src.embeddings.e5_embedder | INFO     | e5_embedder.py:124 | Model loaded, embedding dimension: 1024
2026-02-03 14:42:00 | src.rag.langchain_rag | INFO     | langchain_rag.py:163 | LangChainRAG initialized: model=gpt-4o-mini
Initializing HyDE Retriever...
2026-02-03 14:42:00 | src.rag.hyde         | INFO     | hyde.py:134 | HyDERetriever initialized: blend_factor=0.5
Initializing Reranked RAG...
2026-02-03 14:42:00 | src.rag.reranker     | INFO     | reranker.py:96 | Loading CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-02-03 14:42:01 | src.rag.reranker     | INFO     | reranker.py:98 | CrossEncoder loaded
2026-02-03 14:42:01 | src.rag.reranker     | INFO     | reranker.py:326 | RerankedRAG initialized: reranker=cross_encoder, initial_k=30, final_k=10

✅ All RAG systems successfully initialized


## 2. Method Description

This evaluation compares the following 4 methods:

### A. LLM Only (No RAG) - Baseline
- Directly uses GPT-4o-mini to answer questions
- Does not use any external document retrieval
- Relies solely on the LLM's training knowledge

### B. Simple RAG
- Uses E5 embedding model for vector retrieval of relevant documents
- Supplies the retrieved results as context to the LLM
- The LLM generates answers based on the context

### C. HyDE (Hypothetical Document Embedding)
- First has the LLM generate a "hypothetical document"
- Uses the hypothetical document's embedding for retrieval
- Bridges the semantic gap between the question and the documents

### D. Reranking (Cross-Encoder)
- First stage: Vector retrieval obtains 30 candidate documents
- Second stage: Cross-Encoder re-ranks candidates
- Takes the top-10 as the final context

## 3. Test Questions

Design 5 questions that are challenging for pure LLM:

In [5]:
# Define test questions
test_questions = [
    {
        "id": 1,
        "question": "Is Kevin Durant playing tonight against the Pacers?",
        "team": "HOU",
        "player": "Kevin Durant",
        "description": "Needs real-time injury information for Rockets star",
    },
    {
        "id": 2,
        "question": "What is Brandon Ingram’s injury status today?",
        "team": "TOR",
        "player": "Brandon Ingram",
        "description": "Needs latest status update for Hornets key player",
    },
    {
        "id": 3,
        "question": "Which Timberwolves players are currently listed as questionable or out for tonight?",
        "team": "MIN",
        "player": None,
        "description": "Needs to retrieve information about multiple players",
    },
    {
        "id": 4,
        "question": "What injuries have affected the Grizzlies going into tonight’s game?",
        "team": "MEM",
        "player": None,
        "description": "Needs an overview of the team's injury situation",
    },
    {
        "id": 5,
        "question": "Who are the key NBA players dealing with injuries and questionable statuses for tonight’s slate of games?",
        "team": None,
        "player": None,
        "description": "Needs a cross-team summary for the Feb 2 games",
    },
]


# Display questions
print("Test Question List:")
print("=" * 60)
for q in test_questions:
    print(f"\nQuestion {q['id']}: {q['question']}")
    print(f"   Description: {q['description']}")
    if q['team']:
        print(f"   Team: {q['team']}")
    if q['player']:
        print(f"   Player: {q['player']}")

Test Question List:

Question 1: Is Kevin Durant playing tonight against the Pacers?
   Description: Needs real-time injury information for Rockets star
   Team: HOU
   Player: Kevin Durant

Question 2: What is Brandon Ingram’s injury status today?
   Description: Needs latest status update for Hornets key player
   Team: TOR
   Player: Brandon Ingram

Question 3: Which Timberwolves players are currently listed as questionable or out for tonight?
   Description: Needs to retrieve information about multiple players
   Team: MIN

Question 4: What injuries have affected the Grizzlies going into tonight’s game?
   Description: Needs an overview of the team's injury situation
   Team: MEM

Question 5: Who are the key NBA players dealing with injuries and questionable statuses for tonight’s slate of games?
   Description: Needs a cross-team summary for the Feb 2 games


## 4. Evaluation Results
Execute 4 methods for each question and record the results.

In [6]:
# Define the evaluation function
def evaluate_question(q_info):
    """Evaluate a single question using all methods"""
    question = q_info["question"]
    team = q_info.get("team")
    player = q_info.get("player")
    
    results = {}
    
    # 1. LLM Only
    print("  [1/4] LLM Only...")
    try:
        result = langchain_rag.query_llm_only(question, team=team, player=player)
        results["llm_only"] = {
            "answer": result.answer,
            "sources_count": 0,
        }
    except Exception as e:
        results["llm_only"] = {"error": str(e)}
    
    # 2. Simple RAG
    print("  [2/4] Simple RAG...")
    try:
        result = langchain_rag.query_simple_rag(question, team=team, player=player)
        results["simple_rag"] = {
            "answer": result.answer,
            "sources_count": len(result.sources),
        }
    except Exception as e:
        results["simple_rag"] = {"error": str(e)}
    
    # 3. HyDE
    print("  [3/4] HyDE...")
    try:
        result = hyde_retriever.query(question, team=team, player=player)
        results["hyde"] = {
            "answer": result.answer,
            "sources_count": len(result.sources),
        }
    except Exception as e:
        results["hyde"] = {"error": str(e)}
    
    # 4. Reranking
    print("  [4/4] Reranking...")
    try:
        result = reranked_rag.query(question, team=team, player=player)
        results["rerank"] = {
            "answer": result.answer,
            "sources_count": len(result.sources),
        }
    except Exception as e:
        results["rerank"] = {"error": str(e)}
    
    return results

In [7]:
# Run all evaluations
all_results = {}

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Question {q['id']}: {q['question'][:50]}...")
    print(f"{'='*60}")
    
    all_results[q['id']] = {
        "question": q['question'],
        "description": q['description'],
        "results": evaluate_question(q)
    }
    
print("\n✅ All evaluations completed!")


Question 1: Is Kevin Durant playing tonight against the Pacers...
  [1/4] LLM Only...
  [2/4] Simple RAG...
2026-02-03 14:42:06 | src.vectordb.query   | INFO     | query.py:185 | Query complete: 'Is Kevin Durant playing tonigh...' -> 2 results
  [3/4] HyDE...
  [4/4] Reranking...
2026-02-03 14:42:11 | src.vectordb.query   | INFO     | query.py:185 | Query complete: 'Is Kevin Durant playing tonigh...' -> 2 results

Question 2: What is Brandon Ingram’s injury status today?...
  [1/4] LLM Only...
  [2/4] Simple RAG...
2026-02-03 14:42:15 | src.vectordb.query   | INFO     | query.py:185 | Query complete: 'What is Brandon Ingram’s injur...' -> 0 results
  [3/4] HyDE...
  [4/4] Reranking...
2026-02-03 14:42:18 | src.vectordb.query   | INFO     | query.py:185 | Query complete: 'What is Brandon Ingram’s injur...' -> 0 results

Question 3: Which Timberwolves players are currently listed as...
  [1/4] LLM Only...
  [2/4] Simple RAG...
2026-02-03 14:42:21 | src.vectordb.query   | INFO     | quer

### 4.1 Question 1 Results

In [8]:
# Show results for Question 1
q_id = 1
q_data = all_results[q_id]

print(f"Question: {q_data['question']}")
print(f"Description: {q_data['description']}")
print("\n" + "="*60)

methods = [
    ("llm_only", "A. LLM Only (no RAG)"),
    ("simple_rag", "B. Simple RAG"),
    ("hyde", "C. HyDE RAG"),
    ("rerank", "D. Reranking RAG"),
]

for method_key, method_name in methods:
    print(f"\n### {method_name}")
    result = q_data['results'].get(method_key, {})
    if 'error' in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Source count: {result.get('sources_count', 0)}")
        print(f"\nAnswer:\n{result.get('answer', 'N/A')}")
    print("-" * 40)

Question: Is Kevin Durant playing tonight against the Pacers?
Description: Needs real-time injury information for Rockets star


### A. LLM Only (no RAG)
Source count: 0

Answer:
I'm sorry, but I cannot provide real-time information about player availability or game schedules, as my training data only goes up to October 2023. To find out if Kevin Durant is playing tonight against the Pacers, I recommend checking the latest updates from reliable sports news sources or the official NBA website.
----------------------------------------

### B. Simple RAG
Source count: 2

Answer:
Kevin Durant is not playing tonight against the Pacers, as he is listed as "Out" due to an ankle injury and is expected to be out until at least February 4 (source: [1], [2]).
----------------------------------------

### C. HyDE RAG
Source count: 3

Answer:
Kevin Durant is currently listed as "Out" and is expected to be out until at least February 4 due to an ankle injury (source: [1], [2]). Therefore, he will no

### 4.2 Question 2 Results

In [9]:
# Show results for Question 2
q_id = 2
q_data = all_results[q_id]

print(f"Question: {q_data['question']}")
print(f"Description: {q_data['description']}")
print("\n" + "="*60)

for method_key, method_name in methods:
    print(f"\n### {method_name}")
    result = q_data['results'].get(method_key, {})
    if 'error' in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Source count: {result.get('sources_count', 0)}")
        print(f"\nAnswer:\n{result.get('answer', 'N/A')}")
    print("-" * 40)

Question: What is Brandon Ingram’s injury status today?
Description: Needs latest status update for Hornets key player


### A. LLM Only (no RAG)
Source count: 0

Answer:
As of my last knowledge update in October 2023, Brandon Ingram plays for the New Orleans Pelicans, not the Toronto Raptors (TOR). I do not have access to real-time injury updates or current player statuses. For the latest information on Brandon Ingram's injury status, I recommend checking official team announcements, sports news websites, or the NBA's official channels.
----------------------------------------

### B. Simple RAG
Source count: 0

Answer:
Sorry, no relevant information was found to answer this question.
----------------------------------------

### C. HyDE RAG
Source count: 0

Answer:
Sorry, no relevant information was found to answer this question.
----------------------------------------

### D. Reranking RAG
Source count: 0

Answer:
Sorry, no relevant information found to answer this question.
------

### 4.3 Question 3 Results

In [10]:
# Show results for Question 3
q_id = 3
q_data = all_results[q_id]

print(f"Question: {q_data['question']}")
print(f"Description: {q_data['description']}")
print("\n" + "="*60)

for method_key, method_name in methods:
    print(f"\n### {method_name}")
    result = q_data['results'].get(method_key, {})
    if 'error' in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Source count: {result.get('sources_count', 0)}")
        print(f"\nAnswer:\n{result.get('answer', 'N/A')}")
    print("-" * 40)

Question: Which Timberwolves players are currently listed as questionable or out for tonight?
Description: Needs to retrieve information about multiple players


### A. LLM Only (no RAG)
Source count: 0

Answer:
I'm sorry, but I don't have access to real-time data or updates on player injuries or game statuses as my training only goes up until October 2023. To find the most current information on the Minnesota Timberwolves' roster, including players listed as questionable or out, I recommend checking the official NBA website, the Timberwolves' team site, or a reliable sports news outlet.
----------------------------------------

### B. Simple RAG
Source count: 6

Answer:
Based on the provided information, the following Timberwolves players are currently listed as questionable or out for tonight:

- **Questionable**: 
  - Anthony Edwards (Game Time Decision, Back)
  - Julius Randle (Game Time Decision, Thumb)

- **Out**: 
  - Terrence Shannon Jr. (Expected to be out until at least Feb 4

### 4.4 Question 4 Results

In [11]:
# Show results for Question 4
q_id = 4
q_data = all_results[q_id]

print(f"Question: {q_data['question']}")
print(f"Description: {q_data['description']}")
print("\n" + "="*60)

for method_key, method_name in methods:
    print(f"\n### {method_name}")
    result = q_data['results'].get(method_key, {})
    if 'error' in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Source count: {result.get('sources_count', 0)}")
        print(f"\nAnswer:\n{result.get('answer', 'N/A')}")
    print("-" * 40)

Question: What injuries have affected the Grizzlies going into tonight’s game?
Description: Needs an overview of the team's injury situation


### A. LLM Only (no RAG)
Source count: 0

Answer:
I'm unable to provide real-time updates or the latest injury reports as my training data only goes up to October 2023. For the most accurate and current information regarding injuries affecting the Memphis Grizzlies or any other team, I recommend checking the latest updates from reliable sports news sources or the official NBA website.
----------------------------------------

### B. Simple RAG
Source count: 6

Answer:
The injuries affecting the Grizzlies going into tonight’s game are as follows:

1. **Ja Morant** - Out due to an elbow injury, expected to be out until at least February 20.
2. **Jaren Jackson Jr.** - Game-Time Decision due to a quadriceps injury.
3. **John Konchar** - Expected to be out until at least February 4 due to a neck injury.
4. **Brandon Clarke** - Out, expected to be out

### 4.5 Question 5 Results

In [12]:
# Show results for Question 5
q_id = 5
q_data = all_results[q_id]

print(f"Question: {q_data['question']}")
print(f"Description: {q_data['description']}")
print("\n" + "="*60)

for method_key, method_name in methods:
    print(f"\n### {method_name}")
    result = q_data['results'].get(method_key, {})
    if 'error' in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Source count: {result.get('sources_count', 0)}")
        print(f"\nAnswer:\n{result.get('answer', 'N/A')}")
    print("-" * 40)

Question: Who are the key NBA players dealing with injuries and questionable statuses for tonight’s slate of games?
Description: Needs a cross-team summary for the Feb 2 games


### A. LLM Only (no RAG)
Source count: 0

Answer:
I'm sorry, but I don't have access to real-time data or updates on player injuries or game statuses as of October 2023. For the latest information on key NBA players dealing with injuries or questionable statuses, I recommend checking reliable sports news websites, the NBA's official site, or social media platforms for the most current updates.
----------------------------------------

### B. Simple RAG
Source count: 9

Answer:
Based on the provided context, the key NBA players dealing with injuries and questionable statuses for tonight’s slate of games are:

1. **James Harden** (LAC) - Status: Game Time Decision (Personal) [Source: 3]
2. **Ajay Mitchell** (OKC) - Status: Game Time Decision (Abdomen) [Source: 5]
3. **Stephen Curry** (GSW) - Status: Day-To-Day [S

## 5. Comparative Analysis

In [13]:
# Generate comparison table
import pandas as pd

comparison_data = []

for q_id, q_data in all_results.items():
    row = {
        "Question ID": q_id,
        "Question": q_data['question'][:30] + "...",
    }
    
    for method_key, method_name in methods:
        result = q_data['results'].get(method_key, {})
        if 'error' in result:
            row[method_name.split('. ')[1]] = "Error"
        else:
            sources = result.get('sources_count', 0)
            answer_len = len(result.get('answer', ''))
            row[method_name.split('. ')[1]] = f"Sources:{sources}, Length:{answer_len}"
    
    comparison_data.append(row)

df = pd.DataFrame(comparison_data)
print("\nEvaluation Results Comparison:")
display(df)


Evaluation Results Comparison:


,Question ID,Question,LLM Only (no RAG),Simple RAG,HyDE RAG,Reranking RAG
0,1,Is Kevin Durant playing tonigh...,"Sources:0, Length:318","Sources:2, Length:175","Sources:3, Length:198","Sources:2, Length:159"
1,2,What is Brandon Ingram’s injur...,"Sources:0, Length:372","Sources:0, Length:65","Sources:0, Length:65","Sources:0, Length:61"
2,3,Which Timberwolves players are...,"Sources:0, Length:382","Sources:6, Length:412","Sources:10, Length:319","Sources:6, Length:282"
3,4,What injuries have affected th...,"Sources:0, Length:337","Sources:6, Length:502","Sources:10, Length:452","Sources:6, Length:538"
4,5,Who are the key NBA players de...,"Sources:0, Length:346","Sources:9, Length:460","Sources:10, Length:493","Sources:9, Length:463"


### 5.1 Method Comparison Analysis
**LLM Only vs Simple RAG**
- LLM Only cannot obtain the latest injury information and can only rely on its training data.
- Simple RAG can retrieve relevant documents, providing more accurate and up-to-date information.
**Simple RAG vs HyDE**
- HyDE improves semantic matching between questions and documents by generating hypothetical documents.
- For more complex questions, HyDE may retrieve more relevant documents.
**Simple RAG vs Reranking**
- Reranking uses a Cross-Encoder to rerank the preliminary retrieval results.
- This allows for more accurate identification of the most relevant documents and reduces noise.

## 6. Conclusion

### Key Findings

1. **Value of RAG**: For questions requiring real-time information, RAG is clearly superior to pure LLM.

2. **HyDE Performance**:
   - Pros: Improves semantic matching between questions and documents.
   - Cons: Requires additional LLM calls, which increases latency and cost.

3. **Reranking Performance**:
   - Pros: More accurate document ranking, reduces irrelevant information.
   - Cons: Requires loading an extra Cross-Encoder model.

### Recommendations

- For latency-sensitive applications: Use Simple RAG.
- For applications requiring high accuracy: Use Reranking.
- For complex queries: Consider using HyDE.

In [14]:
# Save results
import json
from datetime import datetime

output_path = project_root / 'outputs' / f'notebook_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"✅ Results have been saved to: {output_path}")

✅ Results have been saved to: /Users/wuyusen/Desktop/Northwestern MLDS/424 Gen AI/hw/nba_lineup_rag/outputs/notebook_results_20260203_144310.json
